<a href="https://colab.research.google.com/github/hridyeshpandey4680-blip/DATA-ANALYTIC/blob/main/Transformer_model(XAI).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install Required Libraries**

In [ ]:
pip install torch transformers pandas numpy scikit-learn shap lime gradio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=73f6c478866817f927993129825bb2903b093e8c73096ff5daaec247c9ab81d8
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime


# Data Exploration & Selection.
# Loading & Inspecting the Datasets in Google Colab.

In [ ]:
import pandas as pd

# Load the primary text + metadata dataset
df_behavioral = pd.read_csv('cyberbullying_dataset.csv.csv')
print("Behavioral Dataset Shape:", df_behavioral.shape)
print(df_behavioral.head(3))

# Load the large Jigsaw text dataset for transformer training
df_jigsaw = pd.read_csv('train.csv.csv', nrows=5000) # Load subset for fast initial testing in Colab
print("\nJigsaw Dataset Shape:", df_jigsaw.shape)
print(df_jigsaw.head(3))

Behavioral Dataset Shape: (1000, 13)
  post_id user_id                                            text  \
0   p0001    u101         Wishing you all the best in your exams.   
1   p0002    u102          You’re a joke, nobody cares about you.   
2   p0003    u103  Happy birthday, hope you have a wonderful day!   

          label  post_length  avg_sentiment  num_likes  num_shares  \
0  non-bullying            8           0.47         69          20   
1      bullying            7          -0.99          5           1   
2  non-bullying            8           0.50         84           8   

   num_reports  user_posts  user_friends  response_time  toxicity_score  
0            0         169           326           3.82            0.12  
1            4         439           107           3.96            0.88  
2            0         172           310           0.74            0.03  

Jigsaw Dataset Shape: (5000, 8)
                 id                                       comment_text  toxi

# **Feature Engineering & Hybrid Model Pipeline**

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel

# 1. Load your behavioral dataset
df = pd.read_csv('cyberbullying_dataset.csv.csv')

# 2. Engineer Additional Behavioral Features
def compute_all_caps_ratio(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return sum(1 for c in text if c.isupper()) / len(text)

df['all_caps_ratio'] = df['text'].apply(compute_all_caps_ratio)

# Map labels to binary (0: non-bullying, 1: bullying)
df['label_binary'] = df['label'].apply(lambda x: 1 if str(x).lower().strip() == 'bullying' else 0)

# Select text and numerical behavioral features
texts = df['text'].tolist()
# Numerical behavioral features: user_posts, user_friends, response_time, num_likes, all_caps_ratio
behavioral_features = df[['user_posts', 'user_friends', 'response_time', 'num_likes', 'all_caps_ratio']].values

labels = df['label_binary'].values

# Train-test split
train_texts, test_texts, train_beh, test_beh, train_labels, test_labels = train_test_split(
    texts, behavioral_features, labels, test_size=0.2, random_state=42
)

print(f"Training samples: {len(train_texts)}, Test samples: {len(test_texts)}")
print("Behavioral feature shape:", behavioral_features.shape)

Training samples: 800, Test samples: 200
Behavioral feature shape: (1000, 5)


# **Initializing the Transformer Encoder (BERT)**

In [ ]:
# Install sentence-transformers or use HuggingFace transformers directly
from transformers import DistilBertTokenizer, DistilBertModel

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')

# Move model to GPU if available in Colab
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)
print("Using device:", device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cuda


# **Building the Hybrid Fusion Model & Training Script**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# 1. Set Device (Force GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")
bert_model.to(device)

# 2. Extract BERT Embeddings (CLS token representation)
def get_bert_embeddings(texts, batch_size=16):
    bert_model.eval()
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = bert_model(**inputs)
            # Use the [CLS] token representation (first token of the last hidden state)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)

    return np.vstack(embeddings)

print("Extracting training text embeddings...")
train_text_emb = get_bert_embeddings(train_texts)
print("Extracting test text embeddings...")
test_text_emb = get_bert_embeddings(test_texts)

# 3. Concatenate Text Embeddings (768-d) with Behavioral Features (5-d) -> Total 773-d
X_train_combined = np.hstack((train_text_emb, train_beh))
X_test_combined = np.hstack((test_text_emb, test_beh))

print(f"Combined Feature Shape (Train): {X_train_combined.shape}")
print(f"Combined Feature Shape (Test): {X_test_combined.shape}")

# 4. Create PyTorch Dataset & DataLoader
class HybridDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = HybridDataset(X_train_combined, train_labels)
test_dataset = HybridDataset(X_test_combined, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 5. Define Hybrid Classifier Network
class HybridClassifier(nn.Module):
    def __init__(self, input_dim):
        super(HybridClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return self.sigmoid(out)

input_dim = X_train_combined.shape[1] # 773
model = HybridClassifier(input_dim).to(device)

# 6. Training Loop
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
print("\nStarting Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

print("\nTraining completed successfully!")

Training on device: cuda
Extracting training text embeddings...
Extracting test text embeddings...
Combined Feature Shape (Train): (800, 773)
Combined Feature Shape (Test): (200, 773)

Starting Training...
Epoch 1/5 | Loss: 0.8480
Epoch 2/5 | Loss: 0.1830
Epoch 3/5 | Loss: 0.0521
Epoch 4/5 | Loss: 0.0408
Epoch 5/5 | Loss: 0.0208

Training completed successfully!


# **Model Evaluation & XAI Integration (SHAP & LIME)**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import shap
import lime
import lime.lime_tabular
import torch

# 1. Evaluate Model on Test Set
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test_combined, dtype=torch.float32).to(device)
    y_pred_probs = model(X_test_tensor).cpu().numpy()
    y_pred = (y_pred_probs >= 0.5).astype(int)

# Calculate Evaluation Metrics
accuracy = accuracy_score(test_labels, y_pred)
precision = precision_score(test_labels, y_pred, zero_division=0)
recall = recall_score(test_labels, y_pred, zero_division=0)
f1 = f1_score(test_labels, y_pred, zero_division=0)

print("=== Model Evaluation Results ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# 2. SHAP Explainability Integration
# We use KernelExplainer or DeepExplainer on a small subset of training background data
print("\nInitializing SHAP Explainer...")
background = X_train_combined[np.random.choice(X_train_combined.shape[0], 50, replace=False)]

def model_predict(np_array):
    tensor_data = torch.tensor(np_array, dtype=torch.float32).to(device)
    with torch.no_grad():
        preds = model(tensor_data).cpu().numpy()
    return preds

explainer = shap.KernelExplainer(model_predict, background)
# Compute SHAP values for a single test sample to verify
shap_values = explainer.shap_values(X_test_combined[:1])
print("SHAP Explainer initialized and test explanation computed successfully!")

# 3. LIME Explainability Integration
print("Initializing LIME Tabular Explainer...")
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_combined,
    feature_names=[f"feat_{i}" for i in range(X_train_combined.shape[1])],
    class_names=['Non-Bullying', 'Bullying'],
    mode='classification'
)

# Explain the first test instance with LIME
exp = explainer_lime.explain_instance(
    X_test_combined[0],
    lambda x: np.hstack((1 - model_predict(x), model_predict(x))),
    num_features=5
)
print("LIME Explainer initialized and test instance explained successfully!")

=== Model Evaluation Results ===
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000

Initializing SHAP Explainer...


  0%|          | 0/1 [00:00<?, ?it/s]

SHAP Explainer initialized and test explanation computed successfully!
Initializing LIME Tabular Explainer...
LIME Explainer initialized and test instance explained successfully!


# **Gradio usage for Web App Interface**

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load a Pre-Trained Toxicity Transformer Encoder
model_name = "unitary/toxic-bert"
print(f"Loading pre-trained toxic semantic model ({model_name})...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
semantic_model = AutoModelForSequenceClassification.from_pretrained(model_name)
semantic_model.to(device)
semantic_model.eval()

# 3. Hybrid Analysis Function with Feature Normalization
def analyze_comment(comment_text, user_posts, user_friends, response_time, num_likes):
    if not comment_text.strip():
        return "Please enter a valid comment.", "0.00%", "No analysis available."

    # Feature 1: Caps Ratio
    caps_ratio = sum(1 for c in comment_text if c.isupper()) / max(len(comment_text), 1)

    # Feature 2: Normalize Behavioral Inputs (0 to 1 scale so they don't dominate)
    norm_posts = min(user_posts / 500.0, 1.0)
    norm_friends = min(user_friends / 1000.0, 1.0)
    norm_caps = caps_ratio

    # Semantic Toxicity Inference
    inputs = tokenizer(comment_text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = semantic_model(**inputs)
        # Apply Sigmoid to get probability output for toxic class
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

    # Extract base text toxicity score (toxic class is index 0 in unitary/toxic-bert)
    text_toxicity_score = float(probs[0])

    # Hybrid Score Fusion: 80% Text Semantics + 20% Behavioral Aggression Signals
    behavioral_risk_score = (norm_caps * 0.5) + (norm_posts * 0.3) + ((1.0 - min(response_time/10.0, 1.0)) * 0.2)
    final_score = (0.80 * text_toxicity_score) + (0.20 * behavioral_risk_score)

    if final_score >= 0.45:
        status = "⚠️ Flagged as Bullying / Toxic"
        confidence = f"{final_score * 100:.2f}%"
    else:
        status = "✅ Classified as Safe / Non-Bullying"
        confidence = f"{(1 - final_score) * 100:.2f}%"

    explanation = (
        f"• Base Text Semantic Toxicity: {text_toxicity_score * 100:.1f}%\n"
        f"• All-Caps Ratio Signal: {caps_ratio:.2f}\n"
        f"• User Post Frequency: {user_posts}\n"
        f"• XAI Fusion Insight: The hybrid system combined deep text semantic probabilities with normalized posting behavior metrics."
    )

    return status, confidence, explanation

# 4. Launch Gradio App
demo = gr.Interface(
    fn=analyze_comment,
    inputs=[
        gr.Textbox(label="Social Media Comment", placeholder="Type comment here..."),
        gr.Slider(1, 500, value=50, step=1, label="User Post Frequency"),
        gr.Slider(0, 1000, value=100, step=1, label="User Friend Count"),
        gr.Slider(0.1, 10.0, value=2.0, step=0.1, label="Response Time (mins)"),
        gr.Slider(0, 100, value=5, step=1, label="Post Likes")
    ],
    outputs=[
        gr.Textbox(label="Prediction Status"),
        gr.Textbox(label="Confidence Score"),
        gr.Textbox(label="XAI Explanation Insights")
    ],
    title="🛡️ Hybrid AI Cyberbullying Detection with Feature Fusion",
    description="Test comments with text semantics and normalized behavioral indicators."
)

demo.launch(share=True)

Loading pre-trained toxic semantic model (unitary/toxic-bert)...


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4b310bc3722fd3fffa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **Gradio usage with voice recognition**

In [ ]:
# 1. Install Whisper and necessary audio dependencies
!pip install -q openai-whisper soundfile

import gradio as gr
import torch
import numpy as np
import whisper
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load Whisper locally for robust speech-to-text
print("Loading Whisper Speech-to-Text model...")
whisper_model = whisper.load_model("tiny")

# Device and Toxicity Model setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
semantic_model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
semantic_model.eval()

def analyze_voice_or_text(audio_file, text_input, user_posts, user_friends, response_time, num_likes):
    comment_text = ""

    # Process Voice Input via Whisper if provided
    if audio_file is not None:
        try:
            audio_path = audio_file if isinstance(audio_file, str) else audio_file[1]
            result = whisper_model.transcribe(audio_path)
            comment_text = result["text"].strip()
        except Exception as e:
            comment_text = ""

    # Fallback or prioritize manual text box if audio transcription is empty
    if not comment_text:
        if text_input and text_input.strip():
            comment_text = text_input.strip()

    if not comment_text:
        return "Please speak into the mic or type a comment.", "0.00%", "No input detected."

    # Behavioral Feature calculations
    caps_ratio = sum(1 for c in comment_text if c.isupper()) / max(len(comment_text), 1)
    norm_posts = min(user_posts / 500.0, 1.0)
    norm_friends = min(user_friends / 1000.0, 1.0)

    # Semantic Toxicity Inference
    inputs = tokenizer(comment_text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = semantic_model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

    text_toxicity_score = float(probs[0])

    # Harassment / Psychological Shaming Keyword Rule Boost
    bullying_keywords = ["never be able", "worthless", "useless", "idiot", "die", "stupid", "worst", "hate", "ugly"]
    keyword_boost = 0.35 if any(kw in comment_text.lower() for kw in bullying_keywords) else 0.0

    behavioral_risk_score = (caps_ratio * 0.5) + (norm_posts * 0.3) + ((1.0 - min(response_time/10.0, 1.0)) * 0.2)
    final_score = (0.60 * text_toxicity_score) + (0.25 * behavioral_risk_score) + keyword_boost
    final_score = min(final_score, 0.99) # Cap at 99%

    # Adjusted Threshold (0.35 for sensitive cyberbullying/harassment detection)
    if final_score >= 0.35:
        status = "⚠️ Flagged as Bullying / Toxic"
        confidence = f"{final_score * 100:.2f}%"
    else:
        status = "✅ Classified as Safe / Non-Bullying"
        confidence = f"{(1 - final_score) * 100:.2f}%"

    explanation = (
        f"• Processed Input Text: \"{comment_text}\"\n"
        f"• Text Semantic Score: {text_toxicity_score * 100:.1f}%\n"
        f"• Harassment Keyword Boost: {'Applied (+35%)' if keyword_boost > 0 else 'None'}\n"
        f"• Behavioral Risk Signal Factor: {behavioral_risk_score * 100:.1f}%\n"
        f"• Final Decision: Combined NLP semantics + behavioral metadata fusion."
    )

    return status, confidence, explanation

# Gradio Interface
demo = gr.Interface(
    fn=analyze_voice_or_text,
    inputs=[
        gr.Audio(sources=["microphone"], type="filepath", label="Voice Input (Speak your comment)"),
        gr.Textbox(label="Type Comment Here", placeholder="Type comment..."),
        gr.Slider(1, 500, value=50, step=1, label="User Post Frequency"),
        gr.Slider(0, 1000, value=100, step=1, label="User Friend Count"),
        gr.Slider(0.1, 10.0, value=2.0, step=0.1, label="Response Time (mins)"),
        gr.Slider(0, 100, value=5, step=1, label="Post Likes")
    ],
    outputs=[
        gr.Textbox(label="Prediction Status"),
        gr.Textbox(label="Confidence Score"),
        gr.Textbox(label="XAI & Behavioral Explanation")
    ],
    title="🎙️🛡️ Multimodal & Voice-Enabled Social Media Behavior Predictor",
    description="Test comments via microphone speech or text typing with enhanced bullying and harassment detection."
)

demo.launch(share=True)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Loading Whisper Speech-to-Text model...


100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 206MiB/s]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3002cb587624649e74.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **Gradio usage for behavioral analysis from social media**

In [ ]:
!pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 27.7 MB/s  0:00:01


In [ ]:
import gradio as gr
import torch
import numpy as np
import speech_recognition as sr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device and Model Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
semantic_model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
semantic_model.eval()

# Custom Emotional & Psychological Bullying Keywords / Patterns
DEMORALIZING_PATTERNS = [
    "cannot do anything", "can't do anything", "good for nothing",
    "worthless", "useless", "loser", "fail in life", "failure",
    "nobody likes you", "waste of space", "ruin everything"
]

def analyze_strict_sandbox(platform, audio_file, text_input, user_posts, engagement_metric, response_time):
    comment_text = ""

    # Handle Text or Voice Input
    if text_input and text_input.strip():
        comment_text = text_input.strip()
    elif audio_file is not None:
        r = sr.Recognizer()
        try:
            if isinstance(audio_file, tuple):
                sr_rate, y = audio_file
                y = y.astype(np.int16)
                import soundfile as sf
                sf.write("temp_audio.wav", y, sr_rate)
                audio_path = "temp_audio.wav"
            else:
                audio_path = audio_file

            with sr.AudioFile(audio_path) as source:
                audio_data = r.record(source)
                comment_text = r.recognize_google(audio_data)
        except Exception:
            comment_text = "[Voice transcription skipped — please type your comment]"

    if not comment_text or comment_text.startswith("[Voice"):
        if not text_input or not text_input.strip():
            return "Please type a comment or provide audio.", "0.00%", "No valid content provided."
        comment_text = text_input.strip()

    # 1. Case-Insensitive Lowercasing for Uniform Matching
    processed_text_lower = comment_text.lower()

    # 2. Semantic Text Analysis with Support for Long/Big Comments (max_length=512)
    inputs = tokenizer(
        processed_text_lower,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = semantic_model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

    text_toxicity_score = float(probs[0]) # Toxic class probability

    # 3. Custom Emotional / Psychological Bullying Pattern Check
    emotional_bully_detected = any(pattern in processed_text_lower for pattern in DEMORALIZING_PATTERNS)
    emotional_boost = 0.45 if emotional_bully_detected else 0.0

    # 4. Subtle Behavioral Booster (Only adjusts confidence by +/- 5% max)
    caps_ratio = sum(1 for c in comment_text if c.isupper()) / max(len(comment_text), 1)
    behavioral_boost = (caps_ratio * 0.05) + (min(user_posts / 500.0, 1.0) * 0.03)

    # Final Score: Text Semantics + Emotional Bullying Boost + Behavioral Adjustment
    final_score = (0.75 * text_toxicity_score) + emotional_boost + behavioral_boost
    final_score = min(max(final_score, 0.0), 1.0) # Clamp between 0 and 1

    # Threshold set at 0.40 for reliable detection
    if final_score >= 0.40:
        status = "⚠️ Flagged as Toxic / Policy Violation"
        confidence = f"{final_score * 100:.2f}%"
        action = "Hide / Flag for Moderator Review"
    else:
        status = f"✅ Approved ({platform} Guidelines Met)"
        confidence = f"{(1 - final_score) * 100:.2f}%"
        action = "Auto-Publish / Safe"

    explanation = (
        f"• Target Platform:- {platform}\n"
        f"• Analyzed Text:- \"{comment_text}\"\n"
        f"• Deep Semantic Toxicity Score:- {text_toxicity_score * 100:.1f}%\n"
        f"• Subtle Emotional Bullying Match:- {'Detected (+45% boost)' if emotional_bully_detected else 'None'}\n"
        f"• Behavioral Risk Adjustment:- +{behavioral_boost * 100:.1f}%\n"
        f"• Recommended Action:- {action}"
    )

    return status, confidence, explanation

# Build Gradio Interface
demo = gr.Interface(
    fn=analyze_strict_sandbox,
    inputs=[
        gr.Dropdown(["Facebook", "YouTube", "Instagram", "WhatsApp"], label="Simulated Social Media Platform", value="YouTube"),
        gr.Audio(sources=["microphone"], type="numpy", label="Voice Input (Optional Speech-to-Text)"),
        gr.Textbox(label="Comment / Message Text (Short or Long)", placeholder="Paste or type comment here..."),
        gr.Slider(1, 500, value=25, step=1, label="Historical Post Frequency"),
        gr.Slider(0, 5000, value=150, step=1, label="Platform Engagement Metric (Subscribers / Followers)"),
        gr.Slider(0.1, 10.0, value=2.0, step=0.1, label="Average Response Interval (mins)")
    ],
    outputs=[
        gr.Textbox(label="Moderation Prediction Status"),
        gr.Textbox(label="Confidence Score"),
        gr.Textbox(label="Detailed XAI & Platform Policy Breakdown")
    ],
    title="🌐 Universal Social Media Moderation Sandbox (High-Accuracy Mode)",
    description="Test comments across platforms with primary semantic toxicity detection and auxiliary behavioral weighting."
)

demo.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://003f2eeab06ac032a9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
